# Experiment 2 · Uncertainty Signal Validation (SageMaker)

**Research question (RQ1 / groundwork for RQ2–RQ3):** can perceptual uncertainty tell us when a detection is wrong, or when an image still contains missed objects?

This notebook **does not train any model**. It reads the saved predictions of an Experiment 1 run (`results/<model_family>/<run_id>/`) and, optionally, runs **one extra horizontally-flipped whole-frame inference** to obtain a second uncertainty signal. Everything else is offline analysis.

What it measures:

1. **Prediction level.** Each prediction is labelled correct (TP) or wrong (FP) with the Experiment 1 rule (class-aware, IoU ≥ 0.5). We test whether uncertainty ranks wrong predictions above correct ones (**AUROC**, with image-level cluster-bootstrap 95% CIs), separately for **base** and **novel** classes.
2. **Calibration.** Does a score of 0.6 really mean 60% precision? **ECE**, **Brier score** and reliability diagrams; then Platt scaling fitted on the **calibration** images and checked on the **policy-test** images.
3. **Beyond object size.** Small boxes are wrong more often, so uncertainty could just be a proxy for size. Logistic models *size only* vs *size + confidence* (+ flip) are fitted on calibration images and evaluated on policy-test images; AUROC of confidence is also reported **within** each size bin.
4. **Image level (missed objects).** Image signals computed from one inference pass (detections just below the threshold, mean uncertainty, entropy, flip disagreement) are correlated with each image's miss rate. These are the signals that Experiment 3 uses to decide where to spend extra computation.

Calibration / policy-test split: deterministic 40 / 60 split of the 548 val images (219 / 329), seed 42. Experiment 3 regenerates the identical split, or can read `image_split.csv` written here.

All complete outputs stay in SageMaker under `results/exp2_uncertainty_validation/<run_id>/`. ClearML receives only metric tables, scalars, figures and a small file inventory, in a **new** project.

In [1]:
# Restart the kernel after the first installation if prompted, then run all cells top-to-bottom.
# ultralytics / pillow are only needed for the optional flip-consistency inference.
%pip install -q "ultralytics>=8.3,<9" "clearml>=1.16,<3" "pillow>=10,<13" "numpy>=1.24,<3" "pandas>=2,<3" "matplotlib>=3.7,<4" "tqdm>=4.66,<5"

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
from getpass import getpass

os.environ["CLEARML_WEB_HOST"] = "https://app.clear.ml"
os.environ["CLEARML_API_HOST"] = "https://api.clear.ml"
os.environ["CLEARML_FILES_HOST"] = "https://files.clear.ml"

os.environ["CLEARML_API_ACCESS_KEY"] = getpass("ClearML access key: ")
os.environ["CLEARML_API_SECRET_KEY"] = getpass("ClearML secret key: ")

print("ClearML credentials set for this kernel.")

ClearML access key:  ········
ClearML secret key:  ········


ClearML credentials set for this kernel.


In [3]:
import csv
import hashlib
import json
import os
import platform
import random
import shutil
import time
import uuid
from collections import Counter, defaultdict
from dataclasses import asdict, dataclass
from datetime import datetime, timezone
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

@dataclass
class Config:
    # The project root is the SAME current directory used by notebook 00 and Experiment 1.
    project_root: str = "SageMaker_VisDrone_OVD_2026"
    task_name_prefix: str = "Experiment 2 - Uncertainty Signal Validation"
    clearml_project: str = "SageMaker-Aerial-OVD-2026/Uncertainty-Signal-Validation-2026"
    run_label: str = "YOLO-E 640 train-ov20"

    # Source run written by the Experiment 1 notebook: results/<source_family>/<source_run_id>/
    source_family: str = "yoloe_26"
    source_run_id: str = "20261028T153524Z_5dd65a8e"
    conditions: tuple = ("zero_shot__whole", "zero_shot__sliced_overlap_20","fine_tuned__whole", "fine_tuned__sliced_overlap_20")

    operating_threshold: float = 0.25        # Same operating threshold as Experiment 1.
    analysis_floor: float = 0.05             # Lowest score analysed for calibration / signals.
    band_low: float = 0.10                   # "Near-miss" band: band_low <= score < operating_threshold.
    evaluation_iou: float = 0.50
    calibration_fraction: float = 0.40       # 548 val images -> 219 calibration / 329 policy-test.
    split_seed: int = 42
    split_csv: str = ""                      # Optional: reuse an existing image_split.csv.
    exclude_early_stopping_images: bool = False  # True: drop the 119 val images used to pick best.pt.
    allow_smoke_sources: bool = False
    ece_bins: int = 15
    bootstrap_samples: int = 1000

    # Optional second signal: horizontal-flip consistency (one extra whole-frame pass per model).
    run_flip_consistency: bool = True
    flip_conditions: tuple = ("zero_shot__whole", "fine_tuned__whole")
    flip_checkpoints: tuple = ("pretrained.pt", "best.pt")   # Under weights/<source_family>/<source_run_id>/
    seed: int = 42

CFG = Config()
assert 0 < CFG.analysis_floor <= CFG.band_low < CFG.operating_threshold < 1
assert 0 < CFG.calibration_fraction < 1 and 0 <= CFG.evaluation_iou <= 1
assert len(CFG.flip_conditions) == len(CFG.flip_checkpoints)
assert all(c.endswith("__whole") for c in CFG.flip_conditions), "Flip consistency is defined for whole-frame conditions."

random.seed(CFG.seed)
np.random.seed(CFG.seed)
ROOT = (Path.cwd().resolve() / CFG.project_root)
DATA_ROOT = ROOT / "visdrone_det_original"
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "_" + uuid.uuid4().hex[:8]
RUN_DIR = ROOT / "runs" / "exp2_uncertainty_validation" / RUN_ID
RESULTS_DIR = ROOT / "results" / "exp2_uncertainty_validation" / RUN_ID
FIGURES_DIR = RESULTS_DIR / "figures"
RUN_DIR.mkdir(parents=True, exist_ok=False)
RESULTS_DIR.mkdir(parents=True, exist_ok=False)
FIGURES_DIR.mkdir()
print("Run:", RUN_ID)
print("Source run:", ROOT / "results" / CFG.source_family / CFG.source_run_id)
print("Persistent local results:", RESULTS_DIR)
print("Free space (GiB):", round(shutil.disk_usage(ROOT).free / 1024**3, 2))
print("New ClearML project:", CFG.clearml_project)

Run: 20261029T102549Z_451ac761
Source run: /home/sagemaker-user/SageMaker_VisDrone_OVD_2026/results/yoloe_26/20261028T153524Z_5dd65a8e
Persistent local results: /home/sagemaker-user/SageMaker_VisDrone_OVD_2026/results/exp2_uncertainty_validation/20261029T102549Z_451ac761
Free space (GiB): 6.62
New ClearML project: SageMaker-Aerial-OVD-2026/Uncertainty-Signal-Validation-2026


## 1 · Load the *published* protocol, not a new random split

Exactly the same checks as the training notebooks: the finalized ClearML Dataset ID in `clearml_protocol_dataset_id.txt` is the only source of category IDs, prompts and the 7-base / 3-novel split. The fingerprint must match before any analysis runs.

In [4]:
from clearml import Dataset, Task

id_file = ROOT / "clearml_protocol_dataset_id.txt"
if not id_file.is_file():
    raise FileNotFoundError(f"Run notebook 00 first; missing {id_file}")
PROTOCOL_ID = id_file.read_text(encoding="utf-8").strip()
ds = Dataset.get(dataset_id=PROTOCOL_ID)
if not ds.is_final():
    raise RuntimeError("Protocol Dataset is not finalized. Do not analyse mutable splits.")
PROTOCOL_DIR = Path(ds.get_local_copy()).resolve()

def sha256_file(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()

required = {"categories_and_prompts.json", "protocol.json", "summary.json",
            "train.csv", "val.csv", "test.csv", "train_base_safe.csv", "protocol_fingerprint.txt"}
if not required.issubset({p.name for p in PROTOCOL_DIR.iterdir() if p.is_file()}):
    raise RuntimeError("ClearML protocol Dataset is missing a required v1 file.")
digest = hashlib.sha256()
for name in sorted(required - {"protocol_fingerprint.txt"}):
    content = (PROTOCOL_DIR / name).read_bytes()
    digest.update(name.encode() + b"\x00" + content + b"\x00")
FINGERPRINT = digest.hexdigest()
if FINGERPRINT != (PROTOCOL_DIR / "protocol_fingerprint.txt").read_text().strip():
    raise RuntimeError("Protocol contents do not match their published fingerprint.")
meta_fingerprint = (ds.get_metadata() or {}).get("protocol_fingerprint")
if meta_fingerprint != FINGERPRINT:
    raise RuntimeError("ClearML Dataset metadata fingerprint does not match its files.")
spec = json.loads((PROTOCOL_DIR / "protocol.json").read_text(encoding="utf-8"))
cats = json.loads((PROTOCOL_DIR / "categories_and_prompts.json").read_text(encoding="utf-8"))
if spec["version"] != "1.0.0":
    raise RuntimeError("This notebook expects the published protocol v1.0.0.")
CATEGORY_IDS = sorted(int(k) for k in cats["official_category_id_to_name"])
CATEGORY_NAMES = [cats["official_category_id_to_name"][str(k)] for k in CATEGORY_IDS]
PROMPTS = [cats["prompt_by_category_name"][name] for name in CATEGORY_NAMES]
BASE_NAMES = cats["base_category_names"]
NOVEL_NAMES = cats["novel_category_names"]
BASE_IDS = set(cats["base_official_ids"])
NOVEL_IDS = set(cats["novel_official_ids"])
assert len(CATEGORY_IDS) == 10 and set(CATEGORY_IDS) == BASE_IDS | NOVEL_IDS
assert BASE_IDS.isdisjoint(NOVEL_IDS) and len(set(PROMPTS)) == 10
GROUPS = {"all": set(CATEGORY_IDS), "base": BASE_IDS, "novel": NOVEL_IDS}

with (PROTOCOL_DIR / "val.csv").open(newline="", encoding="utf-8") as stream:
    val_rows = list(csv.DictReader(stream))
assert len(val_rows) == spec["expected_images"]["val"]
VAL_RELPATHS = {r["image_relpath"] for r in val_rows}
print("Frozen Dataset ID:", PROTOCOL_ID, "| fingerprint:", FINGERPRINT)
print("Base:", BASE_NAMES, "| Novel:", NOVEL_NAMES)

Frozen Dataset ID: 738dce9e1a5440dab34195418a997a20 | fingerprint: 158cb2c904016fa324c236b998393b1a94c176a4eabedb159c76bdeffc66c195
Base: ['pedestrian', 'people', 'bicycle', 'car', 'van', 'truck', 'motor'] | Novel: ['tricycle', 'awning-tricycle', 'bus']


## 3 · Create a NEW ClearML experiment project

No checkpoints or per-prediction files are uploaded: ClearML gets metric tables, scalars, figures and a small inventory with local paths and SHA-256 checksums.

In [5]:
if Task.current_task() is not None:
    raise RuntimeError("An unrelated ClearML task is active. Restart the kernel before this notebook.")

task = Task.init(
    project_name=CFG.clearml_project,
    task_name=f"{CFG.task_name_prefix} | {CFG.run_label} | val | {RUN_ID}",
    reuse_last_task_id=False,
    auto_connect_frameworks=False,
)
logger = task.get_logger()
cfg_logged = asdict(CFG)
task.connect(cfg_logged, name="experiment_config")
task.connect({"protocol_dataset_id": PROTOCOL_ID, "protocol_fingerprint": FINGERPRINT,
              "local_results_dir": str(RESULTS_DIR), "run_id": RUN_ID,
              "python": platform.python_version()}, name="provenance")
print("ClearML task:", task.id)

def record_local_file(path):
    path = Path(path).resolve()
    if not path.is_file() or not path.is_relative_to(RESULTS_DIR):
        raise ValueError(f"Expected a stored file under results/: {path}")
    return {"local_path": str(path), "file_uri": path.as_uri(),
            "bytes": path.stat().st_size, "sha256": sha256_file(path)}

def write_json(filename, value):
    destination = RESULTS_DIR / filename
    if destination.exists():
        raise FileExistsError(f"Refusing to overwrite an earlier output: {destination}")
    destination.write_text(json.dumps(clean_json(value), indent=2, ensure_ascii=False, allow_nan=False) + "\n",
                           encoding="utf-8")
    return destination

def write_csv(filename, frame):
    destination = RESULTS_DIR / filename
    if destination.exists():
        raise FileExistsError(f"Refusing to overwrite an earlier output: {destination}")
    frame.to_csv(destination, index=False)
    return destination

def log_table(title, frame):
    logger.report_table(title=title, series="val", iteration=0, table_plot=frame)

def log_figure(title, series, figure, filename):
    path = FIGURES_DIR / filename
    figure.savefig(path, dpi=150, bbox_inches="tight")
    logger.report_matplotlib_figure(title=title, series=series, figure=figure, iteration=0, report_image=True)
    plt.close(figure)
    return path

LOCAL_INDEX = {"results": {}}

ClearML Task: created new task id=a6db9c83d73343f19540f39f3070771c
ClearML results page: https://app.clear.ml/projects/00e21c7b8f844dc2b776aafa1b827fac/tasks/a6db9c83d73343f19540f39f3070771c/output/log
ClearML task: a6db9c83d73343f19540f39f3070771c
ClearML Monitor: Could not detect iteration reporting, falling back to iterations as seconds-from-start


## 2 · Load saved predictions from earlier runs (no retraining)

Each source run is a folder `results/<model_family>/<run_id>/` written by an Experiment 1 notebook. It must contain `config_and_protocol.json`, `coco_ground_truth.json`, `predictions_<condition>.json` and `latency_<condition>.json`. The loader refuses runs from a different protocol fingerprint, smoke runs (unless explicitly allowed) and runs evaluated on a different image list. Matching follows the Experiment 1 operating rule exactly: class-aware, highest score first, IoU ≥ 0.5.

In [6]:
def read_json(path):
    return json.loads(Path(path).read_text(encoding="utf-8"))

def clean_json(value):
    """Replace NaN/inf with None so json.dumps(allow_nan=False) never fails."""
    if isinstance(value, dict):
        return {str(k): clean_json(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [clean_json(v) for v in value]
    if isinstance(value, (np.floating, float)):
        return None if not np.isfinite(value) else float(value)
    if isinstance(value, np.integer):
        return int(value)
    if isinstance(value, np.bool_):
        return bool(value)
    return value

def size_bin(area):
    if area < 16**2:
        return "tiny"
    if area < 32**2:
        return "small"
    if area < 96**2:
        return "medium"
    return "large"

def group_of(category_id):
    return "base" if category_id in BASE_IDS else "novel"

_SOURCE_CACHE = {}

def load_source_run(family, run_id):
    """Load config, ground truth and the image list of one saved Experiment 1 run."""
    key = (family, run_id)
    if key in _SOURCE_CACHE:
        return _SOURCE_CACHE[key]
    folder = ROOT / "results" / family / run_id
    if not folder.is_dir():
        raise FileNotFoundError(f"Source results folder not found: {folder}")
    src_cfg = read_json(folder / "config_and_protocol.json")
    if src_cfg.get("protocol_fingerprint") != FINGERPRINT:
        raise RuntimeError(f"{folder}: produced with a different protocol fingerprint.")
    run_mode = src_cfg.get("config", {}).get("run_mode")
    if run_mode != "full" and not CFG.allow_smoke_sources:
        raise RuntimeError(f"{folder}: run_mode={run_mode!r}. Smoke results are not paper results; "
                           "set allow_smoke_sources=True only for a pipeline check.")
    if src_cfg.get("config", {}).get("eval_split", "val") != "val":
        raise RuntimeError(f"{folder}: this analysis is a development analysis on val only.")
    coco_gt = read_json(folder / "coco_ground_truth.json")
    id_to_rel = {img["id"]: img["file_name"] for img in coco_gt["images"]}
    rel_to_id = {rel: i for i, rel in id_to_rel.items()}
    eval_rels = list(src_cfg["eval_image_relpaths"])
    if set(eval_rels) != set(id_to_rel.values()):
        raise RuntimeError(f"{folder}: ground-truth images differ from eval_image_relpaths.")
    if not set(eval_rels) <= VAL_RELPATHS:
        raise RuntimeError(f"{folder}: evaluation images are not all in the frozen val manifest.")
    gt = defaultdict(list)
    for ann in coco_gt["annotations"]:
        x, y, w, h = ann["bbox"]
        gt[id_to_rel[ann["image_id"]]].append({
            "gt_id": ann["id"], "category_id": int(ann["category_id"]),
            "xyxy": [x, y, x + w, y + h], "area": float(w * h)})
    source = {"folder": folder, "config": src_cfg, "coco_gt": coco_gt, "id_to_rel": id_to_rel,
              "rel_to_id": rel_to_id, "eval_rels": eval_rels, "gt": gt,
              "early_stopping_rels": set(src_cfg.get("early_stopping_image_relpaths", []))}
    _SOURCE_CACHE[key] = source
    return source

def load_condition(family, run_id, condition):
    """Per-image predictions (xyxy) and per-image latency for one saved condition."""
    source = load_source_run(family, run_id)
    pred_file = source["folder"] / f"predictions_{condition}.json"
    lat_file = source["folder"] / f"latency_{condition}.json"
    for path in (pred_file, lat_file):
        if not path.is_file():
            raise FileNotFoundError(f"Missing saved output: {path}")
    preds = defaultdict(list)
    for p in read_json(pred_file):
        x, y, w, h = p["bbox"]
        preds[source["id_to_rel"][p["image_id"]]].append({
            "category_id": int(p["category_id"]), "score": float(p["score"]),
            "xyxy": [x, y, x + w, y + h], "bbox": [x, y, w, h], "area": float(w * h)})
    latency = {r["image_relpath"]: r for r in read_json(lat_file)}
    missing = set(source["eval_rels"]) - set(latency)
    if missing:
        raise RuntimeError(f"{lat_file}: latency missing for {len(missing)} evaluation images.")
    return {"family": family, "run_id": run_id, "condition": condition, "preds": preds,
            "latency": latency, "source": source}

def iou_matrix(a, b):
    a = np.asarray(a, float).reshape(-1, 4)
    b = np.asarray(b, float).reshape(-1, 4)
    x1 = np.maximum(a[:, None, 0], b[None, :, 0])
    y1 = np.maximum(a[:, None, 1], b[None, :, 1])
    x2 = np.minimum(a[:, None, 2], b[None, :, 2])
    y2 = np.minimum(a[:, None, 3], b[None, :, 3])
    inter = np.clip(x2 - x1, 0, None) * np.clip(y2 - y1, 0, None)
    area_a = np.clip(a[:, 2] - a[:, 0], 0, None) * np.clip(a[:, 3] - a[:, 1], 0, None)
    area_b = np.clip(b[:, 2] - b[:, 0], 0, None) * np.clip(b[:, 3] - b[:, 1], 0, None)
    union = area_a[:, None] + area_b[None, :] - inter
    return np.where(union > 0, inter / np.where(union > 0, union, 1.0), 0.0)

def match_image(preds, gts, floor):
    """Greedy class-aware matching, highest score first, IoU >= CFG.evaluation_iou.

    Because matching is in descending score order, the TP/FP label of every
    prediction above any higher threshold (e.g. 0.25) is identical to what
    Experiment 1 obtained when it matched only predictions above that threshold.
    Returns kept predictions (sorted), their TP flags, and for every GT the score
    of the prediction that matched it (-1 if unmatched).
    """
    keep = sorted((p for p in preds if p["score"] >= floor), key=lambda p: -p["score"])
    tp = np.zeros(len(keep), dtype=bool)
    gt_match_score = np.full(len(gts), -1.0)
    gt_by_class, pred_by_class = defaultdict(list), defaultdict(list)
    for j, g in enumerate(gts):
        gt_by_class[g["category_id"]].append(j)
    for i, p in enumerate(keep):
        pred_by_class[p["category_id"]].append(i)
    for cid, p_idx in pred_by_class.items():
        g_idx = gt_by_class.get(cid, [])
        if not g_idx:
            continue
        ious = iou_matrix([keep[i]["xyxy"] for i in p_idx], [gts[j]["xyxy"] for j in g_idx])
        used = np.zeros(len(g_idx), dtype=bool)
        for row, i in enumerate(p_idx):
            candidate = np.where(used, -1.0, ious[row])
            k = int(np.argmax(candidate))
            if candidate[k] >= CFG.evaluation_iou:
                used[k] = True
                tp[i] = True
                gt_match_score[g_idx[k]] = keep[i]["score"]
    return keep, tp, gt_match_score

def counts_at(keep, tp, gt_match_score, gts, threshold):
    """TP/FP/FN per category group at one confidence threshold (IoU 0.5)."""
    out = {}
    pred_cat = np.array([p["category_id"] for p in keep], dtype=int)
    pred_score = np.array([p["score"] for p in keep], dtype=float)
    gt_cat = np.array([g["category_id"] for g in gts], dtype=int)
    for group, ids in GROUPS.items():
        ids_arr = np.array(sorted(ids))
        pm = (pred_score >= threshold) & np.isin(pred_cat, ids_arr) if len(keep) else np.zeros(0, bool)
        gm = np.isin(gt_cat, ids_arr) if len(gts) else np.zeros(0, bool)
        out[group] = (int((tp & pm).sum()), int((~tp & pm).sum()),
                      int((gm & (gt_match_score < threshold)).sum()))
    return out

def prf(tp, fp, fn):
    P = tp / (tp + fp) if tp + fp else 0.0
    R = tp / (tp + fn) if tp + fn else 0.0
    F = 2 * P * R / (P + R) if P + R else 0.0
    return P, R, F

def image_signals(preds, flip_iou=None):
    """Image-level uncertainty signals computed ONLY from one inference pass.

    n_band       : detections just below the operating threshold (band_low <= s < operating)
    band_ratio   : n_band / (confident detections + 1)
    mean_u       : mean (1 - score) over detections >= analysis_floor
    mean_entropy : mean binary entropy of scores >= analysis_floor
    mean_u_flip  : mean (1 - flip IoU) over confident detections (only when flip data exist)
    small_pred_frac (NOT an uncertainty signal): fraction of predicted boxes < 32^2 px,
                   used as a size-prior baseline.
    """
    # Same ordering as match_image(), so flip_iou (aligned with its `keep`) lines up.
    kept = sorted((p for p in preds if p["score"] >= CFG.analysis_floor), key=lambda p: -p["score"])
    s = np.array([p["score"] for p in kept], dtype=float)
    areas = np.array([p["area"] for p in kept], dtype=float)
    n_conf = int((s >= CFG.operating_threshold).sum())
    n_band = int(((s >= CFG.band_low) & (s < CFG.operating_threshold)).sum())
    eps = 1e-6
    sc = np.clip(s, eps, 1 - eps)
    out = {
        "n_detections": int(s.size),
        "n_confident": n_conf,
        "n_band": n_band,
        "band_ratio": n_band / (n_conf + 1),
        "mean_u": float((1 - s).mean()) if s.size else 1.0,
        "mean_entropy": float((-(sc * np.log(sc) + (1 - sc) * np.log(1 - sc))).mean()) if s.size else float(np.log(2)),
        "small_pred_frac": float((areas < 32**2).mean()) if areas.size else 0.0,
    }
    if flip_iou is not None:
        conf_mask = s >= CFG.operating_threshold
        vals = np.asarray(flip_iou, float)[conf_mask] if conf_mask.size else np.zeros(0)
        out["mean_u_flip"] = float((1 - vals).mean()) if vals.size else 1.0
    return out

def make_split(rels):
    """Deterministic calibration / policy-test split of the evaluation images.

    With 548 val images and calibration_fraction=0.40 this gives 219 / 329.
    If CFG.split_csv points to an earlier split (columns image_relpath, split),
    that file is reused instead so all experiments share one split.
    """
    rels = sorted(rels)
    if CFG.split_csv:
        frame = pd.read_csv(CFG.split_csv)
        mapping = dict(zip(frame["image_relpath"], frame["split"]))
        if set(mapping) != set(rels):
            raise RuntimeError("split_csv does not cover exactly the analysed images.")
        if not set(mapping.values()) <= {"calibration", "policy_test"}:
            raise RuntimeError("split_csv split values must be calibration / policy_test.")
        return mapping
    order = np.random.default_rng(CFG.split_seed).permutation(len(rels))
    n_cal = int(round(len(rels) * CFG.calibration_fraction))
    calibration = {rels[i] for i in order[:n_cal]}
    return {r: ("calibration" if r in calibration else "policy_test") for r in rels}

def auroc(score, label):
    """Probability that a random positive (label 1) has a higher score than a random negative."""
    s = np.asarray(score, float)
    y = np.asarray(label, int)
    ok = np.isfinite(s)
    s, y = s[ok], y[ok]
    n1 = int(y.sum())
    n0 = int(len(y) - n1)
    if n1 == 0 or n0 == 0:
        return float("nan")
    ranks = pd.Series(s).rank(method="average").to_numpy()
    return float((ranks[y == 1].sum() - n1 * (n1 + 1) / 2) / (n1 * n0))

In [7]:
SOURCE = load_source_run(CFG.source_family, CFG.source_run_id)
excluded = SOURCE["early_stopping_rels"] if CFG.exclude_early_stopping_images else set()
ANALYSIS_RELS = sorted(r for r in SOURCE["eval_rels"] if r not in excluded)
SPLIT = make_split(ANALYSIS_RELS)
COND = {c: load_condition(CFG.source_family, CFG.source_run_id, c) for c in CFG.conditions}

MATCH = {}
for c in CFG.conditions:
    MATCH[c] = {rel: match_image(COND[c]["preds"].get(rel, []), SOURCE["gt"].get(rel, []), CFG.analysis_floor)
                for rel in tqdm(ANALYSIS_RELS, desc=f"Match {c}")}

SOURCE_PROVENANCE = [{"family": CFG.source_family, "run_id": CFG.source_run_id,
                      "folder": str(SOURCE["folder"]),
                      "source_clearml_task_id": SOURCE["config"].get("clearml_task_id"),
                      "source_run_mode": SOURCE["config"].get("config", {}).get("run_mode"),
                      "conditions": list(CFG.conditions)}]
split_counts = Counter(SPLIT.values())
print("Analysed images:", len(ANALYSIS_RELS), "| excluded early-stopping images:", len(excluded))
print("Split:", dict(split_counts))
for c in CFG.conditions:
    n = sum(len(MATCH[c][r][0]) for r in ANALYSIS_RELS)
    print(f"{c:22s} predictions >= {CFG.analysis_floor}: {n}")

2026-10-29 10:26:27.316690: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-10-29 10:26:27.335526: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1793269587.358936    1513 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1793269587.366625    1513 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1793269587.384903    1513 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

Match zero_shot__whole:   0%|          | 0/548 [00:00<?, ?it/s]

Match zero_shot__sliced_overlap_20:   0%|          | 0/548 [00:00<?, ?it/s]

Match fine_tuned__whole:   0%|          | 0/548 [00:00<?, ?it/s]

Match fine_tuned__sliced_overlap_20:   0%|          | 0/548 [00:00<?, ?it/s]

Analysed images: 548 | excluded early-stopping images: 0
Split: {'calibration': 219, 'policy_test': 329}
zero_shot__whole       predictions >= 0.05: 26759
zero_shot__sliced_overlap_20 predictions >= 0.05: 38840
fine_tuned__whole      predictions >= 0.05: 59206
fine_tuned__sliced_overlap_20 predictions >= 0.05: 49519


## 4 · Optional second signal: horizontal-flip consistency

A stable detection should reappear in the same place when the image is mirrored. For every saved whole-frame prediction we run the **same checkpoint, prompts, image size, confidence floor and NMS** on the horizontally flipped image, flip the boxes back, and record the best same-class IoU (`flip_iou`, 0 if nothing matches). `u_flip = 1 − flip_iou`; `u_conf_flip = 1 − sqrt(score × flip_iou)` combines both. This costs one extra whole-frame pass per model and needs a GPU kernel; set `run_flip_consistency=False` to skip it.

In [8]:
FLIP_IOU = {}
FLIP_LATENCY = {}

def flip_agreement(keep, flipped_preds):
    out = np.zeros(len(keep))
    by_class = defaultdict(list)
    for q in flipped_preds:
        if q["score"] >= CFG.analysis_floor:
            by_class[q["category_id"]].append(q["xyxy"])
    for i, p in enumerate(keep):
        boxes = by_class.get(p["category_id"])
        if boxes:
            out[i] = float(iou_matrix([p["xyxy"]], boxes).max())
    return out

if CFG.run_flip_consistency:
    import torch
    from PIL import Image
    if not torch.cuda.is_available():
        raise RuntimeError("Flip consistency needs a SageMaker GPU kernel; set run_flip_consistency=False to skip it.")
    YOLO_CONFIG_DIR = RUN_DIR / "ultralytics_config"
    YOLO_CONFIG_DIR.mkdir(exist_ok=True)
    os.environ["YOLO_CONFIG_DIR"] = str(YOLO_CONFIG_DIR)
    from ultralytics import settings, __version__ as ULTRALYTICS_VERSION
    settings.update({"clearml": False})

    src = SOURCE["config"]["config"]
    PRED_KW = dict(imgsz=int(src.get("yolo_image_size", 1280)),
                   conf=float(src.get("yolo_extraction_threshold", 0.001)),
                   iou=float(src.get("yolo_local_nms_iou", 0.70)),
                   max_det=int(src.get("max_detections_per_image", 500)), device=0, verbose=False)
    weights_dir = ROOT / "weights" / CFG.source_family / CFG.source_run_id
    val_sha = {r["image_relpath"]: r["image_sha256"] for r in val_rows}

    def load_model(checkpoint):
        path = weights_dir / checkpoint
        if not path.is_file():
            raise FileNotFoundError(f"Checkpoint not found: {path}")
        if "yoloe" in CFG.source_family.lower():
            from ultralytics import YOLOE
            model = YOLOE(str(path))
            model.set_classes(PROMPTS, model.get_text_pe(PROMPTS))
        else:
            from ultralytics import YOLOWorld
            model = YOLOWorld(str(path))
            model.set_classes(PROMPTS)
        active = [model.names[k] for k in sorted(model.names)]
        if active[:len(PROMPTS)] != PROMPTS:
            raise RuntimeError(f"Prompt order mismatch: expected {PROMPTS!r}, got {active!r}")
        return model

    def predict_whole(model, rgb_array):
        bgr = np.ascontiguousarray(rgb_array[:, :, ::-1])   # PIL RGB -> Ultralytics BGR ndarray
        result = model.predict(source=bgr, **PRED_KW)[0]
        boxes = result.boxes
        if boxes is None or len(boxes) == 0:
            return []
        coords = boxes.xyxy.detach().cpu().numpy()
        classes = boxes.cls.detach().cpu().numpy().astype(int)
        scores = boxes.conf.detach().cpu().numpy()
        return [{"category_id": CATEGORY_IDS[k], "score": float(s), "xyxy": [float(v) for v in b]}
                for b, k, s in zip(coords, classes, scores) if k < len(CATEGORY_IDS)]

    for condition, checkpoint in zip(CFG.flip_conditions, CFG.flip_checkpoints):
        if condition not in CFG.conditions:
            raise ValueError(f"{condition} is not one of the analysed conditions.")
        model = load_model(checkpoint)
        FLIP_IOU[condition], timings, records = {}, [], []
        for rel in tqdm(ANALYSIS_RELS, desc=f"Flip inference {condition}"):
            path = DATA_ROOT / rel
            if sha256_file(path) != val_sha[rel]:
                raise RuntimeError(f"Source image modified since protocol registration: {path}")
            with Image.open(path) as image:
                rgb = np.asarray(image.convert("RGB"))
            W = rgb.shape[1]
            torch.cuda.synchronize()
            start = time.perf_counter()
            with torch.inference_mode():
                flipped = predict_whole(model, np.ascontiguousarray(rgb[:, ::-1]))
            torch.cuda.synchronize()
            timings.append(1000 * (time.perf_counter() - start))
            for q in flipped:                       # Map mirrored boxes back to original coordinates.
                x1, y1, x2, y2 = q["xyxy"]
                q["xyxy"] = [W - x2, y1, W - x1, y2]
                records.append({"image_id": SOURCE["rel_to_id"][rel], "category_id": q["category_id"],
                                "bbox": [W - x2, y1, x2 - x1, y2 - y1], "score": q["score"]})
            FLIP_IOU[condition][rel] = flip_agreement(MATCH[condition][rel][0], flipped)
        FLIP_LATENCY[condition] = float(np.mean(timings))
        write_json(f"flip_predictions_{condition}.json", records)
        print(condition, "flip pass mean ms:", round(FLIP_LATENCY[condition], 2))
        del model
        torch.cuda.empty_cache()
else:
    print("Flip consistency skipped: only confidence- and size-based signals will be analysed.")

WARNING ⚠️ torchvision==0.24 is incompatible with torch==2.8.
Run 'pip install torchvision==0.23' to fix torchvision or 'pip install -U torch torchvision' to update both.
For a full compatibility table see https://github.com/pytorch/vision#installation
Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/home/sagemaker-user/SageMaker_VisDrone_OVD_2026/runs/exp2_uncertainty_validation/20261029T102549Z_451ac761/ultralytics_config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
requirements: Ultralytics requirement ['git+https://github.com/ultralytics/CLIP.git'] not found, attempting AutoUpdate...
Using Python 3.12.15 environment at: /opt/conda
Resolved 18 packages in 744ms
Prepared 2 packages in 2.48s
Installed 2 packages in 15ms
 + clip==1.0 (from git+https://github.com/ultralytics/CLIP.git@7ffa84b3bfa40c42ecc1

Flip inference zero_shot__whole:   0%|          | 0/548 [00:00<?, ?it/s]

zero_shot__whole flip pass mean ms: 89.31


Flip inference fine_tuned__whole:   0%|          | 0/548 [00:00<?, ?it/s]

fine_tuned__whole flip pass mean ms: 36.35


## 5 · Prediction-level validation: does uncertainty rank wrong detections first?

`error = 1` for a false positive, `0` for a true positive. Signals (higher = more uncertain): `u_conf = 1 − score`; `u_size = −log(box area)` (a size-only baseline, not an uncertainty); `u_flip`, `u_conf_flip` when flip data exist. AUROC = 0.5 means no information, 1.0 means perfect ranking. Two prediction sets: **operating** (score ≥ 0.25, what a deployed system outputs) and **analysis floor** (score ≥ 0.05). Raw signals involve no fitting, so they are evaluated on all analysed images; 95% CIs resample whole images, because predictions inside one image are not independent.

In [9]:
rows = []
for c in CFG.conditions:
    flips = FLIP_IOU.get(c)
    for rel in ANALYSIS_RELS:
        keep, tp, _ = MATCH[c][rel]
        fi = flips[rel] if flips is not None else None
        for i, p in enumerate(keep):
            rows.append((c, rel, SPLIT[rel], p["category_id"], group_of(p["category_id"]), p["score"],
                         int(tp[i]), p["area"], size_bin(p["area"]), float(fi[i]) if fi is not None else np.nan))
PRED_DF = pd.DataFrame(rows, columns=["condition", "image_relpath", "split", "category_id", "group", "score",
                                      "correct", "area", "size_bin", "flip_iou"])
PRED_DF["error"] = 1 - PRED_DF["correct"]
PRED_DF["u_conf"] = 1 - PRED_DF["score"]
PRED_DF["u_size"] = -np.log(np.maximum(PRED_DF["area"], 1.0))
PRED_DF["u_flip"] = 1 - PRED_DF["flip_iou"]
PRED_DF["u_conf_flip"] = 1 - np.sqrt(PRED_DF["score"] * PRED_DF["flip_iou"])
SIGNALS = ["u_conf", "u_size"] + (["u_flip", "u_conf_flip"] if FLIP_IOU else [])
THRESHOLDS = {"operating": CFG.operating_threshold, "analysis_floor": CFG.analysis_floor}
rng = np.random.default_rng(CFG.seed)

def cluster_bootstrap_auroc(frame, signal):
    s = frame[signal].to_numpy(float)
    y = frame["error"].to_numpy(int)
    point = auroc(s, y)
    if not np.isfinite(point) or CFG.bootstrap_samples <= 0:
        return point, np.nan, np.nan
    codes, _ = pd.factorize(frame["image_relpath"])
    order = np.argsort(codes, kind="stable")
    counts = np.bincount(codes)
    starts = np.concatenate([[0], np.cumsum(counts)[:-1]])
    groups = [order[a:a + n] for a, n in zip(starts, counts)]
    values = []
    for _ in range(CFG.bootstrap_samples):
        pick = rng.integers(0, len(groups), len(groups))
        idx = np.concatenate([groups[k] for k in pick])
        values.append(auroc(s[idx], y[idx]))
    lo, hi = np.nanpercentile(values, [2.5, 97.5])
    return point, float(lo), float(hi)

auroc_rows = []
for c in CFG.conditions:
    for tname, tvalue in THRESHOLDS.items():
        for group in ("all", "base", "novel"):
            frame = PRED_DF[(PRED_DF["condition"] == c) & (PRED_DF["score"] >= tvalue)]
            if group != "all":
                frame = frame[frame["group"] == group]
            frame = frame.reset_index(drop=True)
            for signal in SIGNALS:
                if frame[signal].notna().sum() == 0:
                    continue
                point, lo, hi = cluster_bootstrap_auroc(frame, signal)
                auroc_rows.append({"condition": c, "prediction_set": tname, "threshold": tvalue, "group": group,
                                   "signal": signal, "n_predictions": len(frame),
                                   "n_errors": int(frame["error"].sum()),
                                   "error_rate": float(frame["error"].mean()) if len(frame) else np.nan,
                                   "AUROC": point, "AUROC_CI_low": lo, "AUROC_CI_high": hi})
AUROC_DF = pd.DataFrame(auroc_rows)

# Confidence AUROC inside each predicted-size bin: if it stays > 0.5, confidence is not just a size proxy.
size_rows = []
for c in CFG.conditions:
    frame = PRED_DF[(PRED_DF["condition"] == c) & (PRED_DF["score"] >= CFG.operating_threshold)]
    for group in ("all", "base", "novel"):
        sub_group = frame if group == "all" else frame[frame["group"] == group]
        for bin_name in ("tiny", "small", "medium", "large"):
            sub = sub_group[sub_group["size_bin"] == bin_name]
            size_rows.append({"condition": c, "group": group, "size_bin": bin_name, "n_predictions": len(sub),
                              "error_rate": float(sub["error"].mean()) if len(sub) else np.nan,
                              "AUROC_u_conf": auroc(sub["u_conf"], sub["error"]),
                              **({"AUROC_u_conf_flip": auroc(sub["u_conf_flip"], sub["error"])} if FLIP_IOU else {})})
AUROC_SIZE_DF = pd.DataFrame(size_rows)
display(AUROC_DF[(AUROC_DF["prediction_set"] == "operating")][
    ["condition", "group", "signal", "n_predictions", "error_rate", "AUROC", "AUROC_CI_low", "AUROC_CI_high"]])

,condition,group,signal,n_predictions,error_rate,AUROC,AUROC_CI_low,AUROC_CI_high
0,zero_shot__whole,all,u_conf,11543,0.289873,0.698195,0.687181,0.709234
1,zero_shot__whole,all,u_size,11543,0.289873,0.557335,0.543912,0.571204
2,zero_shot__whole,all,u_flip,11543,0.289873,0.648153,0.636202,0.660882
3,zero_shot__whole,all,u_conf_flip,11543,0.289873,0.715703,0.704943,0.726245
4,zero_shot__whole,base,u_conf,11101,0.272498,0.696902,0.685880,0.709524
5,zero_shot__whole,base,u_size,11101,0.272498,0.577568,0.563646,0.591819
6,zero_shot__whole,base,u_flip,11101,0.272498,0.654698,0.643294,0.666308
7,zero_shot__whole,base,u_conf_flip,11101,0.272498,0.714622,0.702983,0.725646
8,zero_shot__whole,novel,u_conf,442,0.726244,0.761695,0.704201,0.809380
9,zero_shot__whole,novel,u_size,442,0.726244,0.516671,0.450393,0.580878


## 6 · Calibration: does the score mean what it says?

ECE splits predictions into score bins and averages |precision − mean score| weighted by bin size (0 = perfectly calibrated). Brier = mean (score − correct)². Raw scores are evaluated on all analysed images. **Platt scaling** (a one-feature logistic fit on logit(score)) is fitted only on calibration images and evaluated on policy-test images, showing how much a small calibration set can repair the scores. Uses predictions ≥ `analysis_floor`.

In [10]:
def ece_bins(conf, correct, bins):
    conf = np.asarray(conf, float)
    correct = np.asarray(correct, float)
    edges = np.linspace(0, 1, bins + 1)
    index = np.clip(np.searchsorted(edges, conf, side="right") - 1, 0, bins - 1)
    total, table = 0.0, []
    for b in range(bins):
        m = index == b
        if m.any():
            precision, mean_conf = float(correct[m].mean()), float(conf[m].mean())
            total += m.mean() * abs(precision - mean_conf)
            table.append({"bin": b, "lower": edges[b], "upper": edges[b + 1], "count": int(m.sum()),
                          "mean_confidence": mean_conf, "precision": precision})
    return float(total) if conf.size else np.nan, table

def brier(conf, correct):
    conf, correct = np.asarray(conf, float), np.asarray(correct, float)
    return float(np.mean((conf - correct) ** 2)) if conf.size else np.nan

def logit(p):
    p = np.clip(np.asarray(p, float), 1e-6, 1 - 1e-6)
    return np.log(p / (1 - p))

def fit_logistic(X, y, l2=1e-3, iterations=100):
    """Newton / IRLS logistic regression on standardized features. Returns a predict_proba function."""
    X = np.asarray(X, float).reshape(len(y), -1)
    y = np.asarray(y, float)
    mu, sd = X.mean(0), X.std(0)
    sd[sd == 0] = 1.0
    Z = np.column_stack([np.ones(len(X)), (X - mu) / sd])
    w = np.zeros(Z.shape[1])
    for _ in range(iterations):
        p = 1 / (1 + np.exp(-np.clip(Z @ w, -30, 30)))
        gradient = Z.T @ (p - y) + l2 * np.r_[0, w[1:]]
        hessian = (Z * (p * (1 - p))[:, None]).T @ Z + l2 * np.diag(np.r_[0, np.ones(len(w) - 1)])
        step = np.linalg.solve(hessian + 1e-9 * np.eye(len(w)), gradient)
        w -= step
        if np.abs(step).max() < 1e-8:
            break
    def predict(Xn):
        Zn = np.column_stack([np.ones(len(Xn)), (np.asarray(Xn, float).reshape(len(Xn), -1) - mu) / sd])
        return 1 / (1 + np.exp(-np.clip(Zn @ w, -30, 30)))
    return predict

cal_rows, cal_bin_rows = [], []
for c in CFG.conditions:
    frame = PRED_DF[PRED_DF["condition"] == c]
    cal_part = frame[frame["split"] == "calibration"]
    test_part = frame[frame["split"] == "policy_test"]
    platt = fit_logistic(logit(cal_part["score"]), cal_part["correct"]) if cal_part["correct"].nunique() == 2 else None
    for group in ("all", "base", "novel"):
        f_all = frame if group == "all" else frame[frame["group"] == group]
        f_test = test_part if group == "all" else test_part[test_part["group"] == group]
        ece_all, table = ece_bins(f_all["score"], f_all["correct"], CFG.ece_bins)
        cal_bin_rows.extend({"condition": c, "group": group, **row} for row in table)
        ece_test, _ = ece_bins(f_test["score"], f_test["correct"], CFG.ece_bins)
        row = {"condition": c, "group": group, "n_all": len(f_all),
               "mean_score_all": float(f_all["score"].mean()) if len(f_all) else np.nan,
               "precision_all": float(f_all["correct"].mean()) if len(f_all) else np.nan,
               "ECE_raw_all": ece_all, "Brier_raw_all": brier(f_all["score"], f_all["correct"]),
               "n_policy_test": len(f_test), "ECE_raw_policy_test": ece_test,
               "Brier_raw_policy_test": brier(f_test["score"], f_test["correct"])}
        if platt is not None and len(f_test):
            calibrated = platt(logit(f_test["score"]))
            row["ECE_platt_policy_test"] = ece_bins(calibrated, f_test["correct"], CFG.ece_bins)[0]
            row["Brier_platt_policy_test"] = brier(calibrated, f_test["correct"])
        cal_rows.append(row)
CAL_DF = pd.DataFrame(cal_rows)
CAL_BINS_DF = pd.DataFrame(cal_bin_rows)
display(CAL_DF)

,condition,group,n_all,mean_score_all,precision_all,ECE_raw_all,Brier_raw_all,n_policy_test,ECE_raw_policy_test,Brier_raw_policy_test,ECE_platt_policy_test,Brier_platt_policy_test
0,zero_shot__whole,all,26759,0.287113,0.451400,0.164287,0.213254,16067,0.163575,0.213325,0.012307,0.185029
1,zero_shot__whole,base,25076,0.292523,0.473959,0.181437,0.221103,15056,0.181081,0.221527,0.017743,0.187123
2,zero_shot__whole,novel,1683,0.206509,0.115270,0.092333,0.096309,1011,0.099087,0.091182,0.252424,0.153854
3,zero_shot__sliced_overlap_20,all,38840,0.295848,0.362822,0.067020,0.188198,23220,0.065856,0.190028,0.016062,0.185149
4,zero_shot__sliced_overlap_20,base,34804,0.304852,0.396880,0.092080,0.198991,20790,0.091944,0.201154,0.026551,0.192583
5,zero_shot__sliced_overlap_20,novel,4036,0.218206,0.069128,0.149079,0.095122,2430,0.157334,0.094841,0.228049,0.121554
6,fine_tuned__whole,all,59206,0.320711,0.355184,0.060760,0.114027,34799,0.061634,0.112483,0.016690,0.106076
7,fine_tuned__whole,base,59206,0.320711,0.355184,0.060760,0.114027,34799,0.061634,0.112483,0.016690,0.106076
8,fine_tuned__whole,novel,0,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN
9,fine_tuned__sliced_overlap_20,all,49519,0.418234,0.444597,0.028547,0.142276,29147,0.028959,0.140495,0.020888,0.139868


## 7 · Is uncertainty informative beyond object size?

Logistic models predict `error` for operating-threshold predictions. They are **fitted on calibration images and scored on policy-test images** (AUROC), so this is a held-out test. If *size + confidence* beats *size only*, uncertainty carries information that size alone does not.

In [11]:
FEATURE_SETS = {
    "size_only": ["log_area"],
    "confidence_only": ["logit_score"],
    "size_plus_confidence": ["log_area", "logit_score"],
}
if FLIP_IOU:
    FEATURE_SETS["size_plus_confidence_plus_flip"] = ["log_area", "logit_score", "flip_iou"]

beyond_rows = []
for c in CFG.conditions:
    frame = PRED_DF[(PRED_DF["condition"] == c) & (PRED_DF["score"] >= CFG.operating_threshold)].copy()
    frame["log_area"] = np.log(np.maximum(frame["area"], 1.0))
    frame["logit_score"] = logit(frame["score"])
    for group in ("all", "base", "novel"):
        sub = frame if group == "all" else frame[frame["group"] == group]
        train, test = sub[sub["split"] == "calibration"], sub[sub["split"] == "policy_test"]
        results = {}
        for name, columns in FEATURE_SETS.items():
            if train[columns].isna().any().any() or train["error"].nunique() < 2 or len(test) == 0:
                results[name] = np.nan
                continue
            model = fit_logistic(train[columns].to_numpy(), train["error"].to_numpy())
            results[name] = auroc(model(test[columns].to_numpy()), test["error"])
        beyond_rows.append({"condition": c, "group": group, "n_calibration": len(train), "n_policy_test": len(test),
                            **{f"AUROC_{k}": v for k, v in results.items()},
                            "gain_confidence_over_size": results["size_plus_confidence"] - results["size_only"]})
BEYOND_DF = pd.DataFrame(beyond_rows)
display(BEYOND_DF)

,condition,group,n_calibration,n_policy_test,AUROC_size_only,AUROC_confidence_only,AUROC_size_plus_confidence,AUROC_size_plus_confidence_plus_flip,gain_confidence_over_size
0,zero_shot__whole,all,4429,7114,0.562236,0.697064,0.698422,0.714394,0.136185
1,zero_shot__whole,base,4257,6844,0.579313,0.695390,0.695748,0.711335,0.116435
2,zero_shot__whole,novel,172,270,0.459528,0.772821,0.777923,0.770477,0.318395
3,zero_shot__sliced_overlap_20,all,6814,10385,0.534726,0.673516,0.676711,NaN,0.141986
4,zero_shot__sliced_overlap_20,base,6363,9659,0.562860,0.673809,0.674006,NaN,0.111145
5,zero_shot__sliced_overlap_20,novel,451,726,0.545565,0.662917,0.693723,NaN,0.148158
6,fine_tuned__whole,all,10153,14889,0.646682,0.822027,0.828281,0.830521,0.181599
7,fine_tuned__whole,base,10153,14889,0.646682,0.822027,0.828281,0.830521,0.181599
8,fine_tuned__whole,novel,0,0,NaN,NaN,NaN,NaN,NaN
9,fine_tuned__sliced_overlap_20,all,11667,17182,0.601751,0.783344,0.794962,NaN,0.193211


## 8 · Image level: do uncertainty signals point to images with missed objects?

A detector cannot report what it missed, so we ask whether **image-level** signals from one pass correlate with the image's miss rate (FN / GT at the operating threshold). Spearman ρ > 0 means "more uncertain image → more misses". AUROC asks whether the signal separates images with an above-median miss rate. `small_pred_frac` is a size prior, kept as a baseline. These are exactly the candidate signals for Experiment 3.

In [12]:
IMAGE_SIGNALS = ["n_band", "band_ratio", "mean_u", "mean_entropy", "small_pred_frac"]
image_rows = []
for c in CFG.conditions:
    flips = FLIP_IOU.get(c)
    for rel in ANALYSIS_RELS:
        keep, tp, gms = MATCH[c][rel]
        gts = SOURCE["gt"].get(rel, [])
        counts = counts_at(keep, tp, gms, gts, CFG.operating_threshold)
        signals = image_signals(COND[c]["preds"].get(rel, []), flips[rel] if flips is not None else None)
        n_gt = counts["all"][0] + counts["all"][2]
        n_gt_novel = counts["novel"][0] + counts["novel"][2]
        image_rows.append({"condition": c, "image_relpath": rel, "split": SPLIT[rel], "gt_count": n_gt,
                           "novel_gt_count": n_gt_novel,
                           "TP": counts["all"][0], "FP": counts["all"][1], "FN": counts["all"][2],
                           "novel_FN": counts["novel"][2],
                           "miss_rate": counts["all"][2] / n_gt if n_gt else np.nan,
                           "novel_miss_rate": counts["novel"][2] / n_gt_novel if n_gt_novel else np.nan,
                           "latency_ms": float(COND[c]["latency"][rel]["pipeline_ms"]), **signals})
IMAGE_DF = pd.DataFrame(image_rows)
if FLIP_IOU:
    IMAGE_SIGNALS.insert(4, "mean_u_flip")

corr_rows = []
for c in CFG.conditions:
    frame = IMAGE_DF[IMAGE_DF["condition"] == c]
    for signal in IMAGE_SIGNALS:
        if signal not in frame or frame[signal].notna().sum() < 3:
            continue
        for target in ("miss_rate", "FN", "novel_miss_rate"):
            sub = frame[[signal, target]].dropna()
            rho = sub[signal].corr(sub[target], method="spearman") if len(sub) >= 3 else np.nan
            high = (sub[target] > sub[target].median()).astype(int)
            corr_rows.append({"condition": c, "signal": signal, "target": target, "n_images": len(sub),
                              "spearman_rho": rho,
                              "AUROC_above_median": auroc(sub[signal], high)})
CORR_DF = pd.DataFrame(corr_rows)
display(CORR_DF[CORR_DF["target"] == "miss_rate"])

/opt/conda/lib/python3.12/site-packages/pandas/core/nanops.py:1632: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  return spearmanr(a, b)[0]
/opt/conda/lib/python3.12/site-packages/pandas/core/nanops.py:1632: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  return spearmanr(a, b)[0]
/opt/conda/lib/python3.12/site-packages/pandas/core/nanops.py:1632: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  return spearmanr(a, b)[0]
/opt/conda/lib/python3.12/site-packages/pandas/core/nanops.py:1632: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  return spearmanr(a, b)[0]
/opt/conda/lib/python3.12/site-packages/pandas/core/nanops.py:1632: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  return spearmanr(a, b)[0]
/opt/conda/lib/python3.12/site-packages/pandas/cor

,condition,signal,target,n_images,spearman_rho,AUROC_above_median
0,zero_shot__whole,n_band,miss_rate,548,0.094310,0.551688
3,zero_shot__whole,band_ratio,miss_rate,548,0.539418,0.764712
6,zero_shot__whole,mean_u,miss_rate,548,0.726426,0.834461
9,zero_shot__whole,mean_entropy,miss_rate,548,-0.649187,0.198692
12,zero_shot__whole,mean_u_flip,miss_rate,548,0.251779,0.623502
15,zero_shot__whole,small_pred_frac,miss_rate,548,0.325839,0.669608
18,zero_shot__sliced_overlap_20,n_band,miss_rate,548,0.002008,0.498182
21,zero_shot__sliced_overlap_20,band_ratio,miss_rate,548,0.570062,0.755967
24,zero_shot__sliced_overlap_20,mean_u,miss_rate,548,0.681363,0.813123
27,zero_shot__sliced_overlap_20,mean_entropy,miss_rate,548,-0.595648,0.225465


## 9 · Figures

Reliability diagrams (base vs novel), error rate vs confidence within each size bin, an AUROC summary, and image-level uncertainty vs miss rate. Each figure is saved under `figures/` and logged to ClearML.

In [13]:
FIGURE_FILES = []
COLORS = {"all": "#2a6f97", "base": "#6a994e", "novel": "#bc4749"}

for c in CFG.conditions:
    fig, ax = plt.subplots(figsize=(5, 5))
    ax.plot([0, 1], [0, 1], color="#999999", linestyle="--", linewidth=1, label="perfect calibration")
    for group in ("all", "base", "novel"):
        sub = CAL_BINS_DF[(CAL_BINS_DF["condition"] == c) & (CAL_BINS_DF["group"] == group)]
        if len(sub):
            ece = CAL_DF[(CAL_DF["condition"] == c) & (CAL_DF["group"] == group)]["ECE_raw_all"].iloc[0]
            ax.plot(sub["mean_confidence"], sub["precision"], marker="o", color=COLORS[group],
                    label=f"{group} (ECE {ece:.3f})")
    ax.set(xlim=(0, 1), ylim=(0, 1), xlabel="Mean detector score in bin", ylabel="Precision (IoU ≥ 0.5)",
           title=f"Reliability · {c}")
    ax.legend(frameon=False, fontsize=8)
    FIGURE_FILES.append(log_figure("Reliability diagram", c, fig, f"reliability_{c}.png"))

    frame = PRED_DF[PRED_DF["condition"] == c]
    edges = np.linspace(CFG.analysis_floor, 1.0, 11)
    fig, ax = plt.subplots(figsize=(6, 4))
    for bin_name, color in zip(("tiny", "small", "medium", "large"), ("#bc4749", "#f2a65a", "#6a994e", "#2a6f97")):
        sub = frame[frame["size_bin"] == bin_name]
        index = np.clip(np.searchsorted(edges, sub["score"], side="right") - 1, 0, len(edges) - 2)
        rates = [sub["error"].to_numpy()[index == k].mean() if (index == k).sum() >= 20 else np.nan
                 for k in range(len(edges) - 1)]
        ax.plot((edges[:-1] + edges[1:]) / 2, rates, marker="o", color=color, label=bin_name)
    ax.set(xlabel="Detector score", ylabel="Error rate (FP fraction)", ylim=(0, 1),
           title=f"Error rate by score within size bins · {c}")
    ax.legend(title="predicted box size", frameon=False, fontsize=8)
    FIGURE_FILES.append(log_figure("Error rate vs score by size", c, fig, f"error_vs_score_by_size_{c}.png"))

summary = AUROC_DF[(AUROC_DF["prediction_set"] == "operating") & (AUROC_DF["group"].isin(["base", "novel"]))]
fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
for ax, group in zip(axes, ("base", "novel")):
    sub = summary[summary["group"] == group]
    width = 0.8 / max(1, len(CFG.conditions))
    for k, c in enumerate(CFG.conditions):
        part = sub[sub["condition"] == c].set_index("signal").reindex(SIGNALS)
        x = np.arange(len(SIGNALS)) + k * width
        ax.bar(x, part["AUROC"], width=width, label=c)
        ax.errorbar(x, part["AUROC"], yerr=[part["AUROC"] - part["AUROC_CI_low"], part["AUROC_CI_high"] - part["AUROC"]],
                    fmt="none", ecolor="#333333", capsize=2, linewidth=0.8)
    ax.axhline(0.5, color="#999999", linestyle="--", linewidth=1)
    ax.set_xticks(np.arange(len(SIGNALS)) + width * (len(CFG.conditions) - 1) / 2, SIGNALS, rotation=20)
    ax.set(title=f"{group} classes · score ≥ {CFG.operating_threshold}", ylim=(0.3, 1.0))
axes[0].set_ylabel("AUROC (error detection)")
axes[1].legend(frameon=False, fontsize=7)
FIGURE_FILES.append(log_figure("AUROC summary", "operating", fig, "auroc_summary.png"))

for c in [c for c in CFG.conditions if c.endswith("__whole")]:
    frame = IMAGE_DF[(IMAGE_DF["condition"] == c) & IMAGE_DF["miss_rate"].notna()]
    fig, ax = plt.subplots(figsize=(5, 4))
    ax.scatter(frame["band_ratio"], frame["miss_rate"], s=10, alpha=0.5, color="#2a6f97")
    rho = frame["band_ratio"].corr(frame["miss_rate"], method="spearman")
    ax.set(xlabel="band_ratio (near-miss / confident detections)", ylabel="Image miss rate",
           title=f"Image uncertainty vs misses · {c} (ρ={rho:.2f})")
    FIGURE_FILES.append(log_figure("Image uncertainty vs miss rate", c, fig, f"image_signal_vs_miss_{c}.png"))
print("Figures:", len(FIGURE_FILES))

Figures: 11


## 10 · Keep **all** results locally; log compact tables and the file inventory to ClearML

`results/exp2_uncertainty_validation/<run_id>/` keeps every table, the per-prediction table (`prediction_level.csv.gz`), the per-image table, the split, figures, flip predictions and the configuration. ClearML gets the aggregate tables, headline scalars, figures and a small inventory.

In [14]:
write_csv("auroc.csv", AUROC_DF)
write_csv("auroc_by_size.csv", AUROC_SIZE_DF)
write_csv("calibration.csv", CAL_DF)
write_csv("calibration_bins.csv", CAL_BINS_DF)
write_csv("beyond_size.csv", BEYOND_DF)
write_csv("image_level.csv", IMAGE_DF)
write_csv("image_level_correlations.csv", CORR_DF)
write_csv("prediction_level.csv.gz", PRED_DF)
write_csv("image_split.csv", pd.DataFrame({"image_relpath": list(SPLIT), "split": list(SPLIT.values())}))
write_json("config_and_protocol.json", {"config": asdict(CFG), "run_id": RUN_ID, "clearml_task_id": task.id,
           "protocol_dataset_id": PROTOCOL_ID, "protocol_fingerprint": FINGERPRINT,
           "category_ids": CATEGORY_IDS, "category_names": CATEGORY_NAMES, "prompts": PROMPTS,
           "base_names": BASE_NAMES, "novel_names": NOVEL_NAMES, "sources": SOURCE_PROVENANCE,
           "analysis_image_relpaths": ANALYSIS_RELS, "split_counts": dict(Counter(SPLIT.values())),
           "flip_mean_latency_ms": FLIP_LATENCY, "platform": platform.platform()})

for name, frame in (("AUROC (prediction level)", AUROC_DF), ("AUROC within size bins", AUROC_SIZE_DF),
                    ("Calibration", CAL_DF), ("Beyond size (held-out)", BEYOND_DF),
                    ("Image-level correlations", CORR_DF)):
    log_table(name, frame)

HEADLINE = {}
for _, row in AUROC_DF[AUROC_DF["prediction_set"] == "operating"].iterrows():
    key = f"AUROC_{row['signal']}_{row['group']}"
    logger.report_scalar(title=key, series=row["condition"], value=float(row["AUROC"]), iteration=0)
    HEADLINE[f"{row['condition']}|{key}"] = row["AUROC"]
for _, row in CAL_DF.iterrows():
    for metric in ("ECE_raw_all", "ECE_platt_policy_test", "Brier_raw_all"):
        if metric in row and pd.notna(row[metric]):
            logger.report_scalar(title=f"{metric}_{row['group']}", series=row["condition"], value=float(row[metric]), iteration=0)
            HEADLINE[f"{row['condition']}|{metric}_{row['group']}"] = row[metric]
for _, row in BEYOND_DF.iterrows():
    if pd.notna(row["gain_confidence_over_size"]):
        logger.report_scalar(title=f"gain_confidence_over_size_{row['group']}", series=row["condition"],
                             value=float(row["gain_confidence_over_size"]), iteration=0)
NOTE_TEXT = ("Experiment 2 note: TP/FP use class-aware greedy matching at IoU 0.5 (Experiment 1 rule). "
             "Raw-signal AUROC uses all analysed val images with image-level cluster bootstrap; "
             "Platt and logistic models are fitted on calibration images and evaluated on policy-test images. "
             "Ignore regions are not masked, consistent with Experiment 1.")

In [15]:
for path in sorted(RESULTS_DIR.rglob("*")):
    if path.is_file() and path.name != "local_inventory.json":
        LOCAL_INDEX["results"][path.relative_to(RESULTS_DIR).as_posix()] = record_local_file(path)

inventory = {"run_id": RUN_ID, "experiment": CFG.task_name_prefix,
             "protocol_dataset_id": PROTOCOL_ID, "protocol_fingerprint": FINGERPRINT,
             "clearml_task_id": task.id, "local_results_dir": str(RESULTS_DIR),
             "inventory_path": str(RESULTS_DIR / "local_inventory.json"),
             "sources": SOURCE_PROVENANCE, "objects": LOCAL_INDEX,
             "headline_metrics": HEADLINE}
manifest_file = RESULTS_DIR / "local_inventory.json"
manifest_file.write_text(json.dumps(clean_json(inventory), indent=2, allow_nan=False) + "\n", encoding="utf-8")
if manifest_file.stat().st_size >= 2 * 1024 * 1024:
    raise RuntimeError("ClearML inventory unexpectedly exceeds 2 MiB; inspect before uploading.")
task.upload_artifact(name="local_inventory_small", artifact_object=str(manifest_file))
logger.report_text(f"All complete results saved locally: {RESULTS_DIR}")
logger.report_text(f"Protocol Dataset ID: {PROTOCOL_ID} | SHA256 fingerprint: {FINGERPRINT}")
logger.report_text(NOTE_TEXT)
logger.flush(wait=True)
task.close()
print("DONE. ClearML project:", CFG.clearml_project)
print("ClearML task:", task.id)
print("Local results:", RESULTS_DIR)
print("Free space (GiB):", round(shutil.disk_usage(ROOT).free / 1024**3, 2))
print("IMPORTANT:", "SMOKE SOURCES — pipeline check only, not for the paper" if CFG.allow_smoke_sources
      else "Full-mode sources analysed")

All complete results saved locally: /home/sagemaker-user/SageMaker_VisDrone_OVD_2026/results/exp2_uncertainty_validation/20261029T102549Z_451ac761
Protocol Dataset ID: 738dce9e1a5440dab34195418a997a20 | SHA256 fingerprint: 158cb2c904016fa324c236b998393b1a94c176a4eabedb159c76bdeffc66c195
Experiment 2 note: TP/FP use class-aware greedy matching at IoU 0.5 (Experiment 1 rule). Raw-signal AUROC uses all analysed val images with image-level cluster bootstrap; Platt and logistic models are fitted on calibration images and evaluated on policy-test images. Ignore regions are not masked, consistent with Experiment 1.
DONE. ClearML project: SageMaker-Aerial-OVD-2026/Uncertainty-Signal-Validation-2026
ClearML task: a6db9c83d73343f19540f39f3070771c
Local results: /home/sagemaker-user/SageMaker_VisDrone_OVD_2026/results/exp2_uncertainty_validation/20261029T102549Z_451ac761
Free space (GiB): 6.54
IMPORTANT: Full-mode sources analysed
